In [ ]:
import os
import sys
import random

from collections import defaultdict
from itertools import chain, product
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
import statsmodels.formula.api as smf

from joblib import load
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Patch
from pathos.multiprocessing import ProcessingPool as Pool
from scipy.stats import spearmanr, binom
from tqdm import tqdm, tqdm_notebook

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../")))
import CPC_package as CPC

%matplotlib inline

# Is is more symmetric or asymmetric to build bridges

In [ ]:
#set the random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
def getGraph():
    N, k, beta = 30, 6, 0.1
    #create a WS graph
    graph_A = nx.watts_strogatz_graph(N, k, beta)
    graph_B = nx.watts_strogatz_graph(N, k, beta)
    #relabel all graphs in graph_A by adding the prefix 'A' to the node label
    graph_A = nx.relabel_nodes(graph_A, lambda node: add_prefix(node, 'A'))
    #relabel all graphs in graph_B by adding the prefix 'B' to the node label
    graph_B = nx.relabel_nodes(graph_B, lambda node: add_prefix(node, 'B'))

    #calculate the position of the nodes in the graph_A and graph_B
    pos_A = nx.spring_layout(graph_A, k=0.01, scale=1)
    pos_B = nx.spring_layout(graph_B, k=0.01, scale=1)

    #move every second node inwards 
    for i, node in enumerate(pos_A):
        if i % 2 == 0:
            pos_A[node] *= 0.7
    
    for i, node in enumerate(pos_B):
        if i % 2 == 0:
            pos_B[node] *= 0.7

    #move the nodes of graph_B to the right
    for node in pos_B:
        pos_B[node][0] += 3

    #merge the positions of graph_A and graph_B
    pos = {}
    pos.update(pos_A)
    pos.update(pos_B)

    #merge the graphs into one graph
    graph = nx.compose(graph_A, graph_B)

    #loop overall edges and set a color flag to "black"
    for u, v in graph.edges():
        graph[u][v]['color'] = 'black'

    return graph, pos

def add_prefix(node, prefix):
    return f"{prefix}_{node}"

def plot_the_graphs(graph, pos):
    plt.figure(figsize=(10, 10))
    nx.draw(graph, pos, node_size=20)
    plt.show()

def addRandomTieBetweenGraphs(graph, color='black'):
    #get the nodes of graph_A and graph_B
    nodes_A = [node for node in graph.nodes() if node.startswith('A')]
    nodes_B = [node for node in graph.nodes() if node.startswith('B')]

    success = False

    while not success:
        #pick a random node from graph_A and graph_B
        node_A = random.choice(nodes_A)
        node_B = random.choice(nodes_B)

        #check if there is already a tie between the two nodes
        if not graph.has_edge(node_A, node_B):
            success = True
            #add a tie between the two nodes
            graph.add_edge(node_A, node_B)
            graph[node_A][node_B]['color'] = 'red'

    return graph

def addTriadicClosureBetweenGraphs(graph):
    #get the nodes of graph_A and graph_B
    nodes_A = [node for node in graph.nodes() if node.startswith('A')]
    nodes_B = [node for node in graph.nodes() if node.startswith('B')]

    success = False

    while not success:
        #pick a random node from graph_A and graph_B
        node_A = random.choice(nodes_A)
        node_B = random.choice(nodes_B)

        #check if there is already a tie between the two nodes
        if graph.has_edge(node_A, node_B):
            if random.random() < 0.5:
                #make closure from A to neib of B
                #pick a random neighbor of node_B
                neighbor_B = random.choice(list(graph.neighbors(node_B)))
                #check if there is already a tie between node_A and neighbor_B
                if not graph.has_edge(node_A, neighbor_B) and node_A != neighbor_B:
                    done = True
                    #add a tie between the two nodes
                    graph.add_edge(node_A, neighbor_B)
                    graph[node_A][neighbor_B]['color'] = 'red'
                    success = True
            else:
                #make closure from B to neib of A
                #pick a random neighbor of node_A
                neighbor_A = random.choice(list(graph.neighbors(node_A)))
                #check if there is already a tie between node_B and neighbor_A
                if not graph.has_edge(node_B, neighbor_A) and node_B != neighbor_A:
                    #add a tie between the two nodes
                    graph.add_edge(node_B, neighbor_A)
                    #set the color flag to "red" for this edge
                    graph[neighbor_A][node_B]['color'] = 'red'
                    success = True

    return graph

def plot_the_graphs(graph, pos, filename=None):
    plt.figure(figsize=(5, 2), facecolor='none') 
    ax = plt.gca()
    ax.set_facecolor('none')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

    # 1. Prepare lists for colors and widths based on edge attributes
    edge_colors = []
    edge_widths = []

    for u, v in graph.edges():
        # Get color, default to black if the attribute is missing
        color = graph[u][v].get('color', 'black')
        edge_colors.append(color)
        
        # Set width: 2.0 for red, 1.0 for everything else
        if color == 'red':
            edge_widths.append(1.5)
        else:
            edge_widths.append(1.0)

    # 2. Pass the lists to nx.draw with updated node styling
    nx.draw(
        graph, 
        pos, 
        node_size=50, 
        edge_color=edge_colors, 
        node_color='white',       
        edgecolors='black',       
        linewidths=1.5,           
        width=edge_widths
    )

    if filename is None:
        plt.show()
    else:
        plt.savefig(filename, dpi=300, transparent=True)

In [ ]:
G, pos = getGraph()
G_separated = G.copy()

In [ ]:
G = addRandomTieBetweenGraphs(G, color='red')
for i in range(2):
    G = addTriadicClosureBetweenGraphs(G)
    
plot_the_graphs(G, pos, filename='graph_triadic_3_ties.png')

In [ ]:
for i in range(6):
    G = addTriadicClosureBetweenGraphs(G)
plot_the_graphs(G, pos, filename='graph_triadic_9_ties.png')

In [ ]:
G = G_separated.copy()

In [ ]:
for i in range(6):
    G = addRandomTieBetweenGraphs(G)
plot_the_graphs(G, pos, filename='graph_random_6_ties.png')

In [ ]:
for i in range(12):
    G = addRandomTieBetweenGraphs(G)
plot_the_graphs(G, pos, filename='graph_random_18_ties.png')

In [ ]:
for i in range(17):
    G = addRandomTieBetweenGraphs(G)
plot_the_graphs(G, pos, filename='graph_random_35_ties.png')